In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

df = pd.read_csv('AB_NYC_2019_cleaned.csv')

#Prices above 500 can skew MSE significantly, filtering focuses the model on typical listings
df = df[df['price'] < 500] 
y = df['price']
X = df.drop(columns=['price', 'id', 'name', 'host_id', 'host_name', 'last_review', 'neighbourhood'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ["latitude", "longitude", "minimum_nights", "number_of_reviews", 
                    "reviews_per_month", "calculated_host_listings_count", "availability_365"]
categorical_features = ["neighbourhood_group", "room_type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")), 
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")), 
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ]
)

gb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=100, 
        learning_rate=0.1, 
        max_depth=5, 
        random_state=42
    ))
])

gb_model.fit(X_train, y_train)
y_test_pred = gb_model.predict(X_test)

train_mse = mean_squared_error(y_train, gb_model.predict(X_train))
test_mse = mean_squared_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print(f"Gradient Boosting Results:")
print(f"Train MSE: {train_mse:.2f}")
print(f"Test MSE: {test_mse:.2f}")
print(f"R-squared: {r2:.4f}")

comparison_data = {
    "Model": ["Gradient Boosting"],
    "Train MSE": [train_mse],
    "Test MSE": [test_mse],
    "Notes": ["Uses sequential boosting; handled price cap < 500"]
}
pd.DataFrame(comparison_data).to_csv('gradient_boosting_results.csv', index=False)



Gradient Boosting Results:
Train MSE: 3088.60
Test MSE: 3384.38
R-squared: 0.5237


In [10]:

kf = KFold(n_splits=3, shuffle=True, random_state=42)
#cross validate
cv_mse = -cross_val_score(
    gb_model,
    X_train,
    y_train,
    cv=kf,
    scoring='neg_mean_squared_error'
)

cv_r2 = cross_val_score(
    gb_model,
    X_train,
    y_train,
    cv=kf,
    scoring='r2'
)

print("MSE Scores:", cv_mse)
print("Average CV MSE:", cv_mse.mean())
print("R2 Scores:", cv_r2)
print("Average CV R2:", cv_r2.mean())

gb_model.fit(X_train, y_train)

y_train_pred = gb_model.predict(X_train)
y_test_pred = gb_model.predict(X_test)

train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print("\nFinal Test Results")
print(f"Train MSE: {train_mse:.2f}")
print(f"Test MSE: {test_mse:.2f}")
print(f"Test R2: {r2:.4f}")

comparison_data = {
    "Model": ["Gradient Boosting"],
    "Train MSE": [train_mse],
    "Test MSE": [test_mse],
    "CV Avg MSE": [cv_mse.mean()],
    "CV Avg R2": [cv_r2.mean()],
    "Notes": ["5-Fold Cross Validation + price < 500"]
}

pd.DataFrame(comparison_data).to_csv(
    'gradient_boosting_results.csv',
    index=False
)

Cross Validation Results (5-Fold)
MSE Scores: [3293.52301772 3483.29581382 3425.0800515 ]
Average CV MSE: 3400.632961013544
R2 Scores: [0.53702863 0.52082536 0.53088106]
Average CV R2: 0.529578350996658

Final Test Results
Train MSE: 3088.60
Test MSE: 3384.38
Test R2: 0.5237
